# Coding Assignment 1: Matrices, Systems, and Row Reduction

---

## 🎯 What You'll Learn

By the end of this notebook you will be able to:

- Write Python functions and list comprehensions, and use NumPy arrays confidently
- Tell `@` (matrix product) and `*` (Hadamard product) apart, and know when each is what you want
- Build a matrix product as a sum of **outer products**, and verify $(AB)^\top = B^\top A^\top$ numerically
- Implement **Gauss–Jordan elimination** from scratch and read consistency, pivots, and free variables off the RREF
- Complete your **Week 1 mini-transformer milestone**: turn a sentence into a matrix of embeddings and compute your first (masked) attention score matrix

## ⏱ Estimated Time: 2–3 hours

## 🔗 Relationship to Written HW 1

This notebook is the computational companion to Written HW 1. Several parts here are the *same* facts you proved by hand:

| Written HW 1 | Here |
|---|---|
| Problem 1(b): $(AB)^\top = B^\top A^\top$ | Part 2.2 |
| Problem 1(d): outer products, $AB = \sum_k A_{:k}B_{k:}$ | Parts 2.3, 2.3b |
| Problem 2: row reduction and solution sets | Part 3 |
| Problem 3(c): rescaling by $\mathrm{diag}(v)$ | Part 2.4 |
| Problem 4: redundant feature, RREF, free variables | Part 3.4 |
| Problem 5: softmax and the causal mask | Part 4.4 |

Plenty here is **not** in the written set and is assessed only in this notebook: writing `rref`
yourself, the embedding lookup `PE = X` and its flop count, $P^\top P$ and $PP^\top$, the centering
matrix, and the inconsistent right-hand side.

Doing the written problem first will make the corresponding code cell take about two minutes. Doing it the other way round also works, but is slower.

## 📦 What to Submit

Upload this completed `.ipynb` file to the course portal. All `# TODO` cells must be filled in, and all assertion cells must pass without errors.

You also need `mt_utils.py` in the same folder (see the instructions handout).

---


## Setup


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# mt_utils.py must be in this folder (shared helpers).
import sys, os
sys.path.insert(0, '.')
import mt_utils

np.set_printoptions(precision=4, suppress=True, linewidth=120)
print("NumPy", np.__version__)
print("mt_utils loaded OK")


NumPy 2.5.5
mt_utils loaded OK


---

## Part 1: Python and NumPy Warm-Up

Before the linear algebra, make sure the Python building blocks are comfortable.

### 1.1 Writing a Function

You define a function with `def`:

```python
def square(x):
    return x ** 2

print(square(3))    # 9
```

Functions can return several values at once, as a tuple:

```python
def min_max(lst):
    return min(lst), max(lst)

lo, hi = min_max([3, 1, 4, 1, 5])   # lo = 1, hi = 5
```


In [2]:
# 1.1 Example: a function returning both the sum and the product of a list
def sum_and_product(lst):
    total = 0
    prod = 1
    for x in lst:
        total += x
        prod *= x
    return total, prod

s, p = sum_and_product([1, 2, 3, 4])
print("sum =", s, " product =", p)


sum = 10  product = 24


**Your turn:** Write `count_positives(lst)` returning how many elements of `lst` are **strictly** positive. Note that `0` is not positive.


In [3]:
def count_positives(lst):
    count = 0

    for i in range(len(lst)):
        if lst[i] > 0:
          count += 1

    return count
    pass

# Self-checks
assert count_positives([1, -2, 3, 0, 5]) == 3
assert count_positives([]) == 0
assert count_positives([0, 0, 0]) == 0, "zero is not positive"
assert count_positives([-1, -2]) == 0
print("count_positives OK")


count_positives OK


### 1.2 List Comprehensions

A list comprehension builds a list in one line:

```python
squares = [x**2 for x in range(5)]        # [0, 1, 4, 9, 16]
evens   = [x for x in range(10) if x % 2 == 0]
```

They nest, which is how you build a matrix as a list of lists:

```python
I3 = [[1 if i == j else 0 for j in range(3)] for i in range(3)]
```


In [4]:
# 1.2 Build the n×n identity matrix with a nested list comprehension
n = 4
I = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
for row in I:
    print(row)


[1, 0, 0, 0]
[0, 1, 0, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]


**Your turn:** Write `get_diagonal(M)` that takes a square matrix stored as a list of lists and returns the list of entries $M_{ii}$.


In [5]:
def get_diagonal(M):
    diagonal = []
    for i in range(len(M)):
      diagonal.append(M[i][i])
    return diagonal
    pass

# Self-checks
assert get_diagonal([[1, 2], [3, 4]]) == [1, 4]
assert get_diagonal([[5]]) == [5]
assert get_diagonal([[1,2,3],[4,5,6],[7,8,9]]) == [1, 5, 9]
print("get_diagonal OK")


get_diagonal OK


### 1.3 Indexing: the one-off-by-one that will bite you

Mathematics in this course indexes from **1**. NumPy indexes from **0**.

If $A$ is a matrix, then in math $A_{23}$ is row 2, column 3, but in NumPy that same entry is `A[1, 2]`.
There is no way around this. The only defense is to say out loud which convention you are in.


In [6]:
# 1.3 The 1-based / 0-based gap, made concrete
A = np.array([[10, 20, 30],
              [40, 50, 60],
              [70, 80, 90]])

print("Math A_23  (row 2, col 3, 1-based) =", A[1, 2])   # 60
print("NumPy A[2,3] would be out of bounds for a 3x3 array!")

# Rows and columns.  Math: A_{2:} and A_{:3}.  NumPy:
print("row 2 (1-based)    A[1, :] =", A[1, :])
print("column 3 (1-based) A[:, 2] =", A[:, 2])
print("shape:", A.shape, "-> (rows, columns)")


Math A_23  (row 2, col 3, 1-based) = 60
NumPy A[2,3] would be out of bounds for a 3x3 array!
row 2 (1-based)    A[1, :] = [40 50 60]
column 3 (1-based) A[:, 2] = [30 60 90]
shape: (3, 3) -> (rows, columns)


---

## Part 2: Matrix Products, Transposes, and the Hadamard Product

### 2.1 `@` versus `*`

This is the single most common source of silent bugs in numerical code:

- `A @ B` is the **matrix product**: $(AB)_{ij} = \sum_k a_{ik}b_{kj}$. Needs `A.shape[1] == B.shape[0]`.
- `A * B` is the **Hadamard product**: $(A \odot B)_{ij} = a_{ij}b_{ij}$. Needs identical shapes.

Both run without complaint on square matrices of the same size. They give different answers.


In [7]:
# 2.1 The two products are genuinely different
A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[5., 6.],
              [7., 8.]])

print("A @ B  (matrix product)   =\n", A @ B)
print()
print("A * B  (Hadamard product) =\n", A * B)
print()
print("Equal?", np.allclose(A @ B, A * B))


A @ B  (matrix product)   =
 [[19. 22.]
 [43. 50.]]

A * B  (Hadamard product) =
 [[ 5. 12.]
 [21. 32.]]

Equal? False


### 2.2 The transpose reversal rule

You proved $(AB)^\top = B^\top A^\top$ in Written HW 1, Problem 1(b). Here you check it numerically,
and check that the *wrong* version $A^\top B^\top$ is not even the right shape in general.


In [8]:
# 2.2 Verify (AB)^T = B^T A^T on random non-square matrices
rng = np.random.default_rng(412)
A = rng.standard_normal((3, 5))
B = rng.standard_normal((5, 2))

lhs = (A @ B).T
rhs = B.T @ A.T
print("(AB)^T shape:", lhs.shape, "  B^T A^T shape:", rhs.shape)
print("Equal?", np.allclose(lhs, rhs))

# The order really does have to reverse:
try:
    bad = A.T @ B.T
except ValueError as e:
    print("A^T @ B^T fails, as it must:", e)


(AB)^T shape: (2, 3)   B^T A^T shape: (2, 3)
Equal? True
A^T @ B^T fails, as it must: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


### 2.3 Matrix multiplication as a sum of outer products

From Unit 2:
$$AB \;=\; \sum_{k=1}^{n} A_{:k}\,B_{k:}$$
a sum of rank-one matrices, each formed from the $k$th **column** of $A$ and the $k$th **row** of $B$.

In NumPy, `A[:, k]` is a 1-D array of shape `(n,)`, not a column. To make an outer product you can use
`np.outer(A[:, k], B[k, :])`, which handles the reshaping for you.


**Your turn:** Implement `outer_product_matmul(A, B)` computing $AB$ **only** through the outer-product sum. Do not use `A @ B` or `np.dot`.


In [9]:
def outer_product_matmul(A, B):
    '''Compute A @ B as a sum of outer products of columns of A with rows of B.

    A: (m, n) array,  B: (n, p) array  ->  returns (m, p) array.
    You may use np.outer and np.zeros. Do NOT use @, np.dot, or np.matmul.
    '''
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"inner dimensions must match: {n} != {n2}"

    result = np.zeros((m, p))
    for k in range(n):
      result += np.outer(A[:, k], B[k, :])

    return result

# Self-checks
rng = np.random.default_rng(0)
A = rng.standard_normal((4, 3))
B = rng.standard_normal((3, 5))
assert np.allclose(outer_product_matmul(A, B), A @ B), "does not match A @ B"

A2 = np.array([[1., 2., -2.], [1., 1., -3.]])
B2 = np.array([[-4., 2., 4., -4.], [-1., -5., -3., 3.], [-4., -4., -3., -1.]])
assert np.allclose(outer_product_matmul(A2, B2), A2 @ B2)
print("outer_product_matmul OK")


outer_product_matmul OK


Each term of that sum is a **rank-one** matrix. That is worth seeing directly: a single outer product
$uv^\top$ has every column a multiple of $u$. This is the structure that the SVD (Unit 18) is built on,
and it is why Written HW 1 Problem 1(d) had only one pivot.


In [10]:
# 2.3b One outer product = one rank-one matrix
u = np.array([1., -2.])          # same u, v as Written HW 1, Problem 1(d)
v = np.array([3., 0., 1.])
M = np.outer(u, v)
print("M = u v^T =\n", M)
print("shape:", M.shape, "  rank:", np.linalg.matrix_rank(M))
print("\nEvery column is a multiple of u, so the rank is 1 -- one outer product")
print("carries only one direction's worth of information.")


M = u v^T =
 [[ 3.  0.  1.]
 [-6. -0. -2.]]
shape: (2, 3)   rank: 1

Every column is a multiple of u, so the rank is 1 -- one outer product
carries only one direction's worth of information.


### 2.4 Row and column scaling, and broadcasting

Written HW 1, Problem 3(c) rescales a similarity matrix this way. In general,
$$\mathrm{diag}(r)\,A\,\mathrm{diag}(c) \;=\; A \odot (rc^\top).$$
NumPy lets you do each side without ever building a diagonal matrix, using **broadcasting**:

- `A * c` with `c.shape == (n,)` scales each **column**
- `A * r[:, None]` with `r.shape == (m,)` scales each **row**


In [11]:
# 2.4 Check diag(r) A diag(c) == A * (r c^T), three ways
rng = np.random.default_rng(7)
A = rng.standard_normal((4, 3))
r = np.array([1., 2., 3., 4.])
c = np.array([10., 100., 1000.])

explicit    = np.diag(r) @ A @ np.diag(c)
hadamard    = A * np.outer(r, c)
broadcasted = A * r[:, None] * c

print("diag form == Hadamard form? ", np.allclose(explicit, hadamard))
print("diag form == broadcast form?", np.allclose(explicit, broadcasted))
print("\nBroadcasting never builds the 4x3 matrix r c^T at all -- that is the point.")


diag form == Hadamard form?  True
diag form == broadcast form? True

Broadcasting never builds the 4x3 matrix r c^T at all -- that is the point.


### 2.5 The centering matrix

Before almost any analysis, data is *centered*: each feature's mean is subtracted so every column
averages to zero. It is a small surprise that "subtract the column means" is itself a matrix
multiplication, by
$$H = I_n - \tfrac1n\mathbf{1}\mathbf{1}^\top .$$
This cell has nothing to fill in — just run it and read the output. $H$ turns out to satisfy
$H^\top = H$ and $H^2 = H$, which makes it an *orthogonal projection* (Unit 16), and the same idea
comes back inside every transformer as layer normalization (Unit 19).


In [12]:
# 2.5 Centering: H X = X - 1 mu^T, and H is idempotent
rng = np.random.default_rng(2)
n, d = 6, 3
X_data = rng.standard_normal((n, d)) * 5 + np.array([10., -4., 100.])   # wildly different means

one = np.ones((n, 1))
H = np.eye(n) - (one @ one.T) / n

mu = X_data.T @ one / n                    # (d, 1) column of feature means
print("feature means before centering:", mu.ravel().round(3))
print("feature means after  centering:", (H @ X_data).mean(axis=0).round(12))

print("\nH X == X - 1 mu^T ?", np.allclose(H @ X_data, X_data - one @ mu.T))
print("H^T == H            ?", np.allclose(H.T, H))
print("H @ H == H          ?", np.allclose(H @ H, H))
print("H @ ones == 0       ?", np.allclose(H @ one, 0))
print()
print("Centering twice does nothing new -- that is exactly what H^2 = H says.")


feature means before centering: [  7.034  -1.682 100.458]
feature means after  centering: [ 0. -0.  0.]

H X == X - 1 mu^T ? True
H^T == H            ? True
H @ H == H          ? True
H @ ones == 0       ? True

Centering twice does nothing new -- that is exactly what H^2 = H says.


---

## Part 3: Row Reduction from Scratch

Row reduction is the algorithm behind almost everything in Units 3–4: solving systems, deciding
consistency, finding free variables. NumPy will not do it for you (it has `solve` and `lstsq`, which
are different things), so you will write it.

### 3.1 The three elementary row operations

1. Swap two rows
2. Multiply a row by a nonzero scalar
3. Add a multiple of one row to another

**Gauss–Jordan elimination** applies these until the matrix is in reduced row echelon form (RREF):
each pivot is 1, each pivot is the only nonzero entry in its column, pivots move strictly right as
you go down, and zero rows sit at the bottom.


In [13]:
# 3.1 The three row operations, on a copy
def swap_rows(M, i, j):
    M = M.copy(); M[[i, j]] = M[[j, i]]; return M

def scale_row(M, i, alpha):
    M = M.copy(); M[i] = alpha * M[i]; return M

def add_multiple(M, i, j, alpha):
    '''Row i  <-  Row i + alpha * Row j'''
    M = M.copy(); M[i] = M[i] + alpha * M[j]; return M

M = np.array([[2., 1., -1.],
              [1., 3.,  2.],
              [1., 0.,  0.]])
print("start:\n", M)
print("\nafter swapping rows 1 and 2 (0-based):\n", swap_rows(M, 0, 1))


start:
 [[ 2.  1. -1.]
 [ 1.  3.  2.]
 [ 1.  0.  0.]]

after swapping rows 1 and 2 (0-based):
 [[ 1.  3.  2.]
 [ 2.  1. -1.]
 [ 1.  0.  0.]]


### 3.2 Implementing RREF

**Your turn.** Implement `rref(A)` returning a tuple `(R, pivots)` where `R` is the reduced row
echelon form of `A` and `pivots` is the list of pivot **column indices** (0-based), in order.

The algorithm, one pivot at a time:

```
row = 0
for each column col:
    find the row at or below `row` with the largest |entry| in this column   # partial pivoting
    if that entry is (numerically) zero: continue to the next column
    swap it into position `row`
    scale row `row` so the pivot entry is exactly 1
    subtract multiples of row `row` from EVERY other row to zero out this column
    record col as a pivot; row += 1
```

Two practical notes:

- Work on `A.astype(float).copy()` so you never modify the caller's array.
- Compare against a tolerance, not against `0`. Floating point arithmetic almost never gives exact
  zeros. Use `tol=1e-10`.
- Choosing the largest-magnitude entry as the pivot ("partial pivoting") is what keeps the algorithm
  numerically stable. It is not required for correctness on exact arithmetic, but it is free here.


In [14]:
def rref(A, tol=1e-10):
    '''Reduced row echelon form via Gauss-Jordan elimination.

    Returns (R, pivots):
      R      : (m, n) float array in reduced row echelon form
      pivots : list of 0-based pivot column indices, in increasing order
    '''
    R = np.array(A, dtype=float).copy()
    m, n = R.shape
    pivots = []
    row = 0

    for col in range(n):
        if row >= m:
            break
        #         largest |R[i, col]|.  Hint: np.argmax(np.abs(R[row:, col])) + row
        largest_magnitude_row = np.argmax(np.abs(R[row:, col])) + row

        if np.abs(R[largest_magnitude_row, col]) <= tol:
          continue

        R = swap_rows(R, row, largest_magnitude_row)

        R = scale_row(R, row, 1/R[row, col])

        for i in range(m):
          if i != row:
            R = add_multiple(R, i, row, -R[i, col])

        # These lines were moved from above, to execute after all operations for the current pivot row.
        # This ensures 'row' is at the correct index for scaling and elimination before being incremented.
        pivots.append(col)
        row += 1

        pass

    R[np.abs(R) < tol] = 0.0     # tidy up -0.0 and numerical dust
    return R, pivots

# Self-checks
R, p = rref(np.array([[1., 2.], [3., 4.]]))
assert np.allclose(R, np.eye(2)), f"RREF of an invertible 2x2 should be I, got\n{R}"
assert p == [0, 1]

R, p = rref(np.array([[1., 2., 3.], [2., 4., 6.]]))
assert np.allclose(R, [[1., 2., 3.], [0., 0., 0.]]), f"got\n{R}"
assert p == [0], f"expected pivots [0], got {p}"

R, p = rref(np.array([[0., 0.], [0., 0.]]))
assert np.allclose(R, np.zeros((2, 2))) and p == []

# The matrix M = u v^T from Part 2.3 -- one pivot, as you found by hand
R, p = rref(np.outer([1., 0., -1.], [2., 1., 1.]))
assert p == [0], f"rank-one matrix should have exactly one pivot, got {p}"
assert np.allclose(R[0], [1., 0.5, 0.5]), f"first row should be [1, 0.5, 0.5], got {R[0]}"
print("rref OK")


rref OK


### 3.3 Reading consistency off the RREF

A system $Ax = b$ is **inconsistent** exactly when the RREF of the augmented matrix $[A \mid b]$ has a
pivot in its last column — that is the row $[\,0\ \cdots\ 0 \mid 1\,]$, the equation $0 = 1$.


**Your turn:** Implement `is_consistent(A, b)` using your `rref`.


In [15]:
def is_consistent(A, b):
    '''True if Ax = b has at least one solution.

    Build the augmented matrix [A | b], row reduce it, and check whether the
    last column is a pivot column.
    '''
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    #       whether (n_columns - 1) is NOT among the pivot columns.
    augmented_matrix = np.hstack((A, b))
    R, pivots = rref(augmented_matrix)

    # The last column index of the augmented matrix is augmented_matrix.shape[1] - 1.
    return (augmented_matrix.shape[1] - 1) not in pivots

# Self-checks
assert is_consistent([[1., 1.], [1., -1.]], [2., 0.]) is True or \
       is_consistent([[1., 1.], [1., -1.]], [2., 0.]) == True
assert not is_consistent([[1., 1.], [1., 1.]], [1., 2.]), "x+y=1 and x+y=2 is inconsistent"
assert is_consistent([[1., 1.], [2., 2.]], [1., 2.]), "second equation is redundant, not contradictory"
print("is_consistent OK")


is_consistent OK


### 3.4 Checking your written homework

Your `rref` is now good enough to check both row-reduction problems from Written HW 1. Run it on
Problem 2 (a $3\times3$ system, consistent for one right-hand side and inconsistent for another) and
on Problem 4 (a design matrix with a redundant feature).


In [16]:
# 3.4  Written HW 1, Problem 2 -- consistent vs inconsistent
A2 = np.array([[1., 2., -1.],
               [2., 3.,  1.],
               [1., 1.,  2.]])
b2      = np.array([3., 5., 2.])    # Problem 2(a)
b2_bad  = np.array([3., 5., 3.])    # Problem 2(c): only the last entry changed

R, piv = rref(np.hstack([A2, b2.reshape(-1, 1)]))
print("Problem 2(a):  RREF of [A | b] =\n", R)
print("  pivots:", piv, "  consistent?", is_consistent(A2, b2),
      "  free variables:", A2.shape[1] - len([c for c in piv if c < A2.shape[1]]))

R2, piv2 = rref(np.hstack([A2, b2_bad.reshape(-1, 1)]))
print("\nProblem 2(c):  RREF of [A | b'] =\n", R2)
print("  pivots:", piv2, "  consistent?", is_consistent(A2, b2_bad))
print("  Column 3 (0-based) is the augmented column, and it is a pivot column:")
print("  that last row reads 0 = 1. Changing one number on the right-hand side")
print("  turned a system with infinitely many solutions into one with none.")


Problem 2(a):  RREF of [A | b] =
 [[ 1.  0.  5.  1.]
 [ 0.  1. -3.  1.]
 [ 0.  0.  0.  0.]]
  pivots: [0, 1]   consistent? True   free variables: 1

Problem 2(c):  RREF of [A | b'] =
 [[ 1.  0.  5.  0.]
 [ 0.  1. -3.  0.]
 [ 0.  0.  0.  1.]]
  pivots: [0, 1, 3]   consistent? False
  Column 3 (0-based) is the augmented column, and it is a pivot column:
  that last row reads 0 = 1. Changing one number on the right-hand side
  turned a system with infinitely many solutions into one with none.


In [17]:
# 3.4b Written HW 1, Problem 4 -- the redundant feature z3 = z1 + z2
A4 = np.array([[1., 1., 0., 1.],
               [1., 0., 1., 1.],
               [1., 1., 1., 2.]])
b4 = np.array([3., 4., 6.])

R4, piv4 = rref(np.hstack([A4, b4.reshape(-1, 1)]))
print("RREF of [A | b] =\n", R4)
print("pivots:", piv4, "  free variables:", A4.shape[1] - len(piv4))

n = np.array([0., -1., -1., 1.])
print("\nA n =", A4 @ n, " (n spans the homogeneous solutions)")
print("column 4 == column 2 + column 3 ?", np.allclose(A4[:, 3], A4[:, 1] + A4[:, 2]),
      " <- this IS the relation z3 = z1 + z2")

p = np.array([1., 2., 3., 0.])
print()
for t in [0.0, 1.0, -2.5]:
    w = p + t * n
    print(f"t = {t:5.2f}   w = {np.round(w, 3)}   A w = {np.round(A4 @ w, 6)}")
print()
print("Different weights, identical predictions: the coefficients are not identifiable.")


RREF of [A | b] =
 [[1. 0. 0. 0. 1.]
 [0. 1. 0. 1. 2.]
 [0. 0. 1. 1. 3.]]
pivots: [0, 1, 2]   free variables: 1

A n = [0. 0. 0.]  (n spans the homogeneous solutions)
column 4 == column 2 + column 3 ? True  <- this IS the relation z3 = z1 + z2

t =  0.00   w = [1. 2. 3. 0.]   A w = [3. 4. 6.]
t =  1.00   w = [1. 1. 2. 1.]   A w = [3. 4. 6.]
t = -2.50   w = [ 1.   4.5  5.5 -2.5]   A w = [3. 4. 6.]

Different weights, identical predictions: the coefficients are not identifiable.


---

## Part 4: 🤖 Mini-Transformer Milestone

### Week 1 Milestone — Tokens, Embeddings, and a First Score Matrix

This is where the semester-long project starts. Every later assignment adds to `mt_utils.py`; by the
end you will have a complete mini-transformer, and you will have written all of it.

**The setup.** A transformer never sees text. It sees a sequence of integer token IDs, and its very
first act is to replace each ID by a vector — a row of the *embedding matrix* $E$.

Our toy vocabulary:

| Token | ID |
|---|---|
| `[BOS]` | 0 |
| `the` | 1 |
| `cat` | 2 |
| `sat` | 3 |
| `mat` | 4 |
| `[EOS]` | 5 |

(These IDs are 0-based, because they are array indices.)


In [18]:
# 4.1 Build the embedding matrix
mt_utils.set_seed(412)

VOCAB = ['[BOS]', 'the', 'cat', 'sat', 'mat', '[EOS]']
VOCAB_SIZE = len(VOCAB)
D_MODEL = 8          # embedding dimension; small enough to print

E = mt_utils.make_embedding_matrix(VOCAB_SIZE, D_MODEL)
print("Embedding matrix E has shape", E.shape, "= (vocab_size, d_model)")
print("Row v of E is the embedding of token v.\n")
print(E.round(4))


Embedding matrix E has shape (6, 8) = (vocab_size, d_model)
Row v of E is the embedding of token v.

[[-0.0186  0.      0.0053 -0.0371 -0.0061 -0.0094 -0.0017 -0.0055]
 [ 0.0033  0.0034  0.0516 -0.0163  0.0039  0.0172 -0.0036 -0.001 ]
 [ 0.0247  0.025   0.0385 -0.0089 -0.0459 -0.0584  0.0025 -0.0185]
 [ 0.0002  0.0034 -0.017  -0.0163  0.0154  0.0043  0.0159 -0.0395]
 [-0.0124 -0.0214 -0.0253 -0.0145  0.0013 -0.0241  0.0168 -0.0375]
 [ 0.0118 -0.012  -0.014  -0.0066 -0.009  -0.0046  0.0256 -0.0038]]


### 4.2 Embedding lookup is a matrix product

If $P$ is the one-hot matrix of the sequence, then
$$PE = X,$$
the matrix whose $t$th row is the embedding of the $t$th token: multiplying by a one-hot row *selects*
a row of $E$. Here you verify that the identity holds — and then count how absurdly wasteful it would
be to actually compute it that way.


**Your turn:** Build the one-hot matrix `P` for a list of token IDs.


In [19]:
def one_hot_matrix(token_ids, vocab_size):
    '''Return P of shape (L, vocab_size) with P[t, token_ids[t]] = 1 and zeros elsewhere.'''
    L = len(token_ids)
    P = np.zeros((L, vocab_size))
    v = np.arange(L)
    P[v, token_ids] = 1

    return P

# "the cat sat" -> IDs [1, 2, 3]
sentence_ids = [1, 2, 3]
P = one_hot_matrix(sentence_ids, VOCAB_SIZE)
print("P =\n", P)

X_matmul = P @ E                                   # the honest matrix product
X_lookup = mt_utils.embed_tokens(sentence_ids, E)  # what a real implementation does

assert np.allclose(X_matmul, X_lookup), "P @ E should equal the row lookup"
print("\nP @ E equals the direct lookup:", np.allclose(X_matmul, X_lookup))
print("X has shape", X_lookup.shape, "= (sequence_length, d_model)")


P =
 [[0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]]

P @ E equals the direct lookup: True
X has shape (3, 8) = (sequence_length, d_model)


In [20]:
# 4.2b Why nobody computes it that way.  Realistic sizes:
V, L, d = 50_000, 512, 512
matmul_flops = L * V * d
lookup_copies = L * d
print(f"P @ E   : {matmul_flops:,} multiply-adds")
print(f"lookup  : {lookup_copies:,} scalar copies")
print(f"ratio   : {matmul_flops // lookup_copies:,}x   (this is exactly V)")
print()
print("Storing P alone would take", f"{L*V:,}", "entries to hold", L, "ones.")

# Two more products worth knowing: P^T P counts how often each token occurs,
# and P P^T marks which pairs of positions hold the SAME token.
P_repeat = one_hot_matrix([2, 3, 2, 5], VOCAB_SIZE)
print("\nP^T P (token frequency histogram on the diagonal):")
print(P_repeat.T @ P_repeat)
print("\nP P^T (1 exactly where two positions hold the same token):")
print(P_repeat @ P_repeat.T)


P @ E   : 13,107,200,000 multiply-adds
lookup  : 262,144 scalar copies
ratio   : 50,000x   (this is exactly V)

Storing P alone would take 25,600,000 entries to hold 512 ones.

P^T P (token frequency histogram on the diagonal):
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 2. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1.]]

P P^T (1 exactly where two positions hold the same token):
[[1. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 0. 0. 1.]]


### 4.3 A first score matrix

An attention head computes $S = \tfrac{1}{\sqrt{d_k}}QK^\top$ with $Q = XW_Q$ and $K = XW_K$.
Because $Q = XW_Q$ and $K = XW_K$, the reversal rule of Written HW 1, Problem 1(b) gives
$QK^\top = X(W_QW_K^\top)X^\top$ — the head depends on $W_Q$ and $W_K$ only through the single product
$M = W_QW_K^\top$.

You will build real $W_Q$ and $W_K$ in a later assignment. For now take $W_Q = W_K = I$, so that
$M = I$ and $S$ is just the matrix of scaled dot products between token embeddings — a first, honest
attention score matrix.


In [21]:
# 4.3 Score matrix, computed two ways
X = X_lookup
d_k = D_MODEL

W_Q = np.eye(D_MODEL)
W_K = np.eye(D_MODEL)

Q = X @ W_Q
K = X @ W_K
S_direct = (Q @ K.T) / np.sqrt(d_k)

M = W_Q @ W_K.T
S_via_M = (X @ M @ X.T) / np.sqrt(d_k)

print("S (direct)  =\n", S_direct.round(4))
print("\nSame via X M X^T?", np.allclose(S_direct, S_via_M))
print("S has shape", S_direct.shape, "= (L, L) -- one score per ordered pair of tokens")


S (direct)  =
 [[ 0.0012  0.0004 -0.0002]
 [ 0.0004  0.0031 -0.0002]
 [-0.0002 -0.0002  0.0009]]

Same via X M X^T? True
S has shape (3, 3) = (L, L) -- one score per ordered pair of tokens


### 4.4 The causal mask

A model that generates text left to right must not let a token see the future. The mask $\Omega$
records *which* positions are permitted: $\Omega_{ij} = 1$ when position $i$ may attend to position $j$.

**How the mask gets applied is the subtle part**, and Written HW 1, Problem 5 is where you work it
out. The tempting move — zero out the forbidden scores with $S \odot \Omega$ — does *not* work:
a $0$ still passes through the softmax as $e^0 = 1$, so a forbidden position would collect real
attention weight. The cell after next uses the procedure that does work: put $-\infty$ in the
forbidden entries *before* the softmax, so they exponentiate to exactly $0$.

**Your turn:** build the mask.


In [22]:
def causal_mask(L):
    '''Return the L x L matrix with 1 on and below the diagonal, 0 above.

    Entry (i, j) is 1 exactly when j <= i, i.e. position i may attend to position j.
    Hint: np.tril(np.ones((L, L))) does this in one call -- but try the
    list-comprehension version first, then check them against each other.
    '''
    return np.tril(np.ones((L, L)))
    pass

Omega = causal_mask(4)
print("Causal mask (L=4):\n", Omega)
assert np.allclose(Omega, np.tril(np.ones((4, 4))))
assert causal_mask(1).shape == (1, 1)

# A score matrix to mask:
S_hw = np.array([[2., 1., 0., 3.],
                 [0., 4., 1., 1.],
                 [1., 1., 2., 0.],
                 [3., 0., 1., 5.]])
print("\nS (*) Omega =\n", S_hw * causal_mask(4))
print("\nThis displays WHICH entries are forbidden -- it does not enforce anything.")
print("A forbidden score of 0 would exponentiate to 1 and collect real weight.")
print("The next cell does the real thing.")


Causal mask (L=4):
 [[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]

S (*) Omega =
 [[2. 0. 0. 0.]
 [0. 4. 0. 0.]
 [1. 1. 2. 0.]
 [3. 0. 1. 5.]]

This displays WHICH entries are forbidden -- it does not enforce anything.
A forbidden score of 0 would exponentiate to 1 and collect real weight.
The next cell does the real thing.


In [23]:
# 4.4b Masking our own score matrix, and a first look at attention weights
L = X.shape[0]
mask = causal_mask(L)

# The real thing: -inf in the forbidden entries BEFORE the softmax, so they
# exponentiate to exactly 0.  Contrast with S * mask above, where a forbidden
# score of 0 would exponentiate to 1 and collect real attention weight.
S_masked = np.where(mask == 1, S_direct, -np.inf)     # additive -inf mask (Method 2)

def softmax_rows(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)          # numerical stability
    ex = np.exp(Z)
    return ex / np.sum(ex, axis=1, keepdims=True)

Aw = softmax_rows(S_masked)
print("Attention weights (each row sums to 1, upper triangle is 0):\n", Aw.round(4))
print("\nrow sums:", Aw.sum(axis=1).round(6))
print()
for t, tid in enumerate(sentence_ids):
    share = ", ".join(f"{VOCAB[sentence_ids[j]]}={Aw[t, j]:.2f}" for j in range(t + 1))
    print(f"  '{VOCAB[tid]}' attends to: {share}")


Attention weights (each row sums to 1, upper triangle is 0):
 [[1.     0.     0.    ]
 [0.4993 0.5007 0.    ]
 [0.3332 0.3332 0.3336]]

row sums: [1. 1. 1.]

  'the' attends to: the=1.00
  'cat' attends to: the=0.50, cat=0.50
  'sat' attends to: the=0.33, cat=0.33, sat=0.33


Look closely at those weights: apart from the mask, they are almost perfectly **uniform**
(0.50/0.50, then 0.33/0.33/0.33). That is not a bug, and it is worth understanding.

`make_embedding_matrix` scales its random entries by 0.02, so the dot products in $S$ are on the
order of $10^{-3}$. Since $e^x \approx 1 + x$ for small $x$, a softmax over nearly-equal scores
returns nearly-equal weights. **An untrained attention head attends almost uniformly to everything
it is allowed to see.** Learning to attend *selectively* is precisely what training accomplishes, and
you will watch it happen in the final project.

If you want to see it now, re-run the cell above with `E = mt_utils.make_embedding_matrix(...) * 50`
and watch the weights spread out. (Don't leave that change in your submission.)


### ✅ Milestone Checklist

Before submitting, verify:

- [ ] `count_positives`, `get_diagonal`, `outer_product_matmul`, `rref`, `is_consistent`,
      `one_hot_matrix`, `causal_mask` all pass their assertion cells
- [ ] Your `rref` reproduces the RREF you computed by hand in Written HW 1, Problems 2(a) and 4(a)
- [ ] `P @ E` matches the direct embedding lookup
- [ ] Every row of the masked attention weight matrix sums to 1, and the upper triangle is zero
- [ ] You can say, in one sentence, why `A @ B` and `A * B` are different

### 🧠 One Question to Sit With

In Part 4.4 the first token attends only to itself, so its attention weight is 1.00 regardless of what
the scores were. Is that a bug or is it correct? (It is correct. Be sure you see why.)


---

## 🤖 AI Use Statement (required)

Course policy allows AI assistance and requires you to cite it. In the cell below, briefly state:
what you asked an assistant, what it gave you, and what you changed. If you used none, write "None."


*(None. I used DeepML.org to help me in the coding of rref)*
